In [ ]:
import math
import random
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import Tensor, nn


In [ ]:
# Physical, geometric, source, and output parameters shared with the FEM notebook.
from params import (
    A_stim,
    T,
    a1,
    a2,
    hole_r,
    hole_x,
    hole_y,
    r_value,
    s_stim,
    snapshot_times,
    source_amplitude,
    source_period,
    source_phase,
    source_spatial_width,
    source_time_width,
    source_x,
    source_y,
    x0,
    y0,
)


In [ ]:
results_dir = Path("comparison_results")
results_dir.mkdir(exist_ok=True)

In [ ]:
@dataclass(frozen=True)
class ModelSpec:
    hidden_layers: int
    hidden_width: int


class MonodomainPINN(nn.Module):
    """MLP approximation with an output transform enforcing u(x, y, 0)=u0."""

    def __init__(self, spec: ModelSpec) -> None:
        super().__init__()
        layers: list[nn.Module] = []
        input_width = 3
        for _ in range(spec.hidden_layers):
            linear = nn.Linear(input_width, spec.hidden_width)
            nn.init.xavier_normal_(linear.weight)
            nn.init.zeros_(linear.bias)
            layers.extend((linear, nn.Tanh()))
            input_width = spec.hidden_width
        output = nn.Linear(input_width, 1)
        nn.init.xavier_normal_(output.weight)
        nn.init.zeros_(output.bias)
        layers.append(output)
        self.network = nn.Sequential(*layers)

    def forward(self, points: Tensor) -> Tensor:
        x = points[:, 0:1]
        y = points[:, 1:2]
        t = points[:, 2:3]
        # x and y already lie in [-1, 1]; normalize time to the same interval.
        normalized = torch.cat((x, y, 2.0 * t / T - 1.0), dim=1)
        raw = self.network(normalized)

        # This transform is exactly u0 at t=0 while leaving the network free
        # to represent the solution for every t>0.
        reaction_decay = torch.exp(-r_value * t)
        return reaction_decay * initial_condition(x, y) + (1.0 - reaction_decay) * raw


## FEM-aligned anisotropic reaction-diffusion problem

The network approximates $u(x,y,t)$ on the unit disk with the circular hole

$$\Omega=\{(x,y):x^2+y^2<1\}\setminus
\{(x,y):(x-x_h)^2+(y-y_h)^2\leq r_h^2\}.$$

It solves the same continuous PDE as `FEM-code-new-simple.ipynb`:

$$u_t-\nabla\cdot(D\nabla u)+r u=F(x,y,t),\qquad
D=\operatorname{diag}(a_1,a_2),$$

with the Gaussian initial condition from `params.py` and homogeneous Neumann
conditions $(D\nabla u)\cdot n=0$ on both the outer circle and the hole. The
forcing is $F(x,y,t)=f(x,y)g(t)$, using the same spatial Gaussian and smooth
periodic Gaussian-like heartbeat as the FEM notebook.


In [ ]:
def initial_condition(x: Tensor, y: Tensor) -> Tensor:
    radius_sq = (x - x0) ** 2 + (y - y0) ** 2
    return A_stim * torch.exp(-radius_sq / (2.0 * s_stim**2))


def diffusivity(x: Tensor, y: Tensor) -> Tensor:
    """Return the diagonal entries of the constant anisotropic tensor D."""
    return torch.cat((torch.full_like(x, a1), torch.full_like(y, a2)), dim=1)


def source_spatial_profile(x: Tensor, y: Tensor) -> Tensor:
    radius_sq = (x - source_x) ** 2 + (y - source_y) ** 2
    return source_amplitude * torch.exp(
        -radius_sq / (2.0 * source_spatial_width**2)
    )


def source_time_profile(t: Tensor) -> Tensor:
    phase_angle = math.pi * (t - source_phase) / source_period
    scaled_width = math.pi * source_time_width / source_period
    return torch.exp(-torch.sin(phase_angle).square() / (2.0 * scaled_width**2))


def source_term(x: Tensor, y: Tensor, t: Tensor) -> Tensor:
    return source_spatial_profile(x, y) * source_time_profile(t)


def sample_times(
    count: int,
    device: torch.device,
    dtype: torch.dtype,
    early_time_fraction: float,
) -> Tensor:
    uniform_draw = torch.rand((count, 1), device=device, dtype=dtype)
    early_draw = torch.rand((count, 1), device=device, dtype=dtype)
    choose_early = torch.rand((count, 1), device=device) < early_time_fraction
    # Squaring a uniform draw concentrates samples near the rapid initial transient.
    return T * torch.where(choose_early, early_draw.square(), uniform_draw)


def sample_interior(
    count: int,
    device: torch.device,
    dtype: torch.dtype,
    early_time_fraction: float,
) -> Tensor:
    """Uniformly sample the unit disk, rejecting points in the circular hole."""
    if count <= 0:
        raise ValueError("count must be positive")

    accepted: list[Tensor] = []
    remaining = count
    while remaining:
        candidate_count = 2 * remaining
        radius = torch.sqrt(
            torch.rand((candidate_count, 1), device=device, dtype=dtype)
        )
        angle = 2.0 * math.pi * torch.rand(
            (candidate_count, 1), device=device, dtype=dtype
        )
        xy = torch.cat((radius * torch.cos(angle), radius * torch.sin(angle)), dim=1)
        outside_hole = (
            (xy[:, 0] - hole_x).square() + (xy[:, 1] - hole_y).square()
            >= hole_r**2
        )
        batch = xy[outside_hole][:remaining]
        accepted.append(batch)
        remaining -= len(batch)

    xy = torch.cat(accepted, dim=0)
    t = sample_times(count, device, dtype, early_time_fraction)
    return torch.cat((xy, t), dim=1)


def sample_boundary(
    count: int,
    device: torch.device,
    dtype: torch.dtype,
    early_time_fraction: float,
) -> tuple[Tensor, Tensor]:
    """Sample both circles by perimeter and return points plus outward normals."""
    if count <= 0:
        raise ValueError("count must be positive")

    # Circumferences are proportional to radius: outer radius 1, inner radius hole_r.
    inner_count = int(round(count * hole_r / (1.0 + hole_r)))
    if count >= 2:
        inner_count = min(max(inner_count, 1), count - 1)
    outer_count = count - inner_count

    outer_angle = 2.0 * math.pi * torch.rand(
        (outer_count, 1), device=device, dtype=dtype
    )
    outer_direction = torch.cat(
        (torch.cos(outer_angle), torch.sin(outer_angle)), dim=1
    )
    outer_xy = outer_direction
    outer_normals = outer_direction

    inner_angle = 2.0 * math.pi * torch.rand(
        (inner_count, 1), device=device, dtype=dtype
    )
    inner_direction = torch.cat(
        (torch.cos(inner_angle), torch.sin(inner_angle)), dim=1
    )
    hole_center = torch.tensor(
        [[hole_x, hole_y]], device=device, dtype=dtype
    )
    inner_xy = hole_center + hole_r * inner_direction
    # The outward normal of the perforated domain points into the hole.
    inner_normals = -inner_direction

    xy = torch.cat((outer_xy, inner_xy), dim=0)
    normals = torch.cat((outer_normals, inner_normals), dim=0)
    t = sample_times(count, device, dtype, early_time_fraction)
    points = torch.cat((xy, t), dim=1)

    permutation = torch.randperm(count, device=device)
    return points[permutation], normals[permutation]


In [ ]:
def gradient(value: Tensor, points: Tensor, create_graph: bool = True) -> Tensor:
    return torch.autograd.grad(
        value,
        points,
        grad_outputs=torch.ones_like(value),
        create_graph=create_graph,
        retain_graph=True,
    )[0]


def pde_residual(model: MonodomainPINN, points: Tensor) -> Tensor:
    points = points.detach().requires_grad_(True)
    u = model(points)
    grad_u = gradient(u, points)
    u_x, u_y, u_t = grad_u[:, 0:1], grad_u[:, 1:2], grad_u[:, 2:3]
    a = diffusivity(points[:, 0:1], points[:, 1:2])
    a1_values, a2_values = a[:, 0:1], a[:, 1:2]

    flux_x, flux_y = a1_values * u_x, a2_values * u_y
    div_flux = (
        gradient(flux_x, points)[:, 0:1]
        + gradient(flux_y, points)[:, 1:2]
    )
    forcing = source_term(
        points[:, 0:1], points[:, 1:2], points[:, 2:3]
    )
    return u_t - div_flux + r_value * u - forcing


def boundary_flux(
    model: MonodomainPINN, points: Tensor, normals: Tensor
) -> Tensor:
    points = points.detach().requires_grad_(True)
    u = model(points)
    grad_u = gradient(u, points)
    a = diffusivity(points[:, 0:1], points[:, 1:2])
    a1_values, a2_values = a[:, 0:1], a[:, 1:2]
    return (
        a1_values * grad_u[:, 0:1] * normals[:, 0:1]
        + a2_values * grad_u[:, 1:2] * normals[:, 1:2]
    )


In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")    

In [ ]:
dtype = torch.float32 #torch.float64
if device.type == "mps" and dtype == torch.float64:
    raise ValueError("MPS does not support float64; use --dtype float32")
torch.set_default_dtype(dtype)

In [ ]:
set_seed(42)

In [ ]:
hidden_layers       = 5
hidden_width        = 128
learning_rate       = 1.0e-3
epochs              = 12000
min_learning_rate   = 1.0e-5
#
interior_batch      = 8192 #4096 #2048
boundary_batch      = 512
early_time_fraction = 0.5
#
boundary_weight     = 10.0
grad_clip           = 1.0
log_every           = 100

In [ ]:
spec = ModelSpec(hidden_layers, hidden_width)
model = MonodomainPINN(spec).to(device=device, dtype=dtype)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=epochs, eta_min=min_learning_rate
)


In [ ]:
def validate_setup(
    model: MonodomainPINN,
    device: torch.device,
    dtype: torch.dtype,
    early_time_fraction: float,
    count: int = 64,
) -> float:
    """Lightweight checks for geometry, source, residuals, and autodiff."""
    interior = sample_interior(count, device, dtype, early_time_fraction)
    boundary, normals = sample_boundary(count, device, dtype, early_time_fraction)
    assert interior.shape == (count, 3)
    assert boundary.shape == (count, 3)
    assert normals.shape == (count, 2)

    tolerance = 100.0 * torch.finfo(dtype).eps
    interior_outer_radius_sq = interior[:, 0].square() + interior[:, 1].square()
    interior_hole_radius_sq = (
        (interior[:, 0] - hole_x).square()
        + (interior[:, 1] - hole_y).square()
    )
    assert torch.all(interior_outer_radius_sq <= 1.0 + tolerance)
    assert torch.all(interior_hole_radius_sq >= hole_r**2 - tolerance)

    boundary_outer_radius = torch.linalg.vector_norm(boundary[:, :2], dim=1)
    boundary_hole_radius = torch.linalg.vector_norm(
        boundary[:, :2]
        - torch.tensor([hole_x, hole_y], device=device, dtype=dtype),
        dim=1,
    )
    on_outer = torch.isclose(
        boundary_outer_radius,
        torch.ones_like(boundary_outer_radius),
        atol=tolerance,
        rtol=tolerance,
    )
    on_hole = torch.isclose(
        boundary_hole_radius,
        torch.full_like(boundary_hole_radius, hole_r),
        atol=tolerance,
        rtol=tolerance,
    )
    assert torch.all(on_outer | on_hole)
    assert torch.any(on_outer) and torch.any(on_hole)
    assert torch.allclose(
        torch.linalg.vector_norm(normals, dim=1),
        torch.ones(count, device=device, dtype=dtype),
        atol=tolerance,
        rtol=tolerance,
    )

    expected_normals = torch.empty_like(normals)
    expected_normals[on_outer] = boundary[on_outer, :2]
    hole_center = torch.tensor([hole_x, hole_y], device=device, dtype=dtype)
    expected_normals[on_hole] = (
        hole_center - boundary[on_hole, :2]
    ) / hole_r
    assert torch.allclose(
        normals, expected_normals, atol=tolerance, rtol=tolerance
    )

    source_peak = source_term(
        torch.full((1, 1), source_x, device=device, dtype=dtype),
        torch.full((1, 1), source_y, device=device, dtype=dtype),
        torch.full((1, 1), source_phase, device=device, dtype=dtype),
    )
    assert torch.allclose(
        source_peak,
        torch.full_like(source_peak, source_amplitude),
        atol=tolerance,
        rtol=tolerance,
    )

    residual = pde_residual(model, interior)
    flux = boundary_flux(model, boundary, normals)
    assert residual.shape == (count, 1) and torch.isfinite(residual).all()
    assert flux.shape == (count, 1) and torch.isfinite(flux).all()

    # Exercise automatic differentiation through both loss terms.
    smoke_loss = residual.square().mean() + flux.square().mean()
    parameter_gradients = torch.autograd.grad(smoke_loss, tuple(model.parameters()))
    assert all(torch.isfinite(value).all() for value in parameter_gradients)

    initial_points = sample_interior(count, device, dtype, early_time_fraction)
    initial_points[:, 2] = 0.0
    with torch.inference_mode():
        initial_error = (
            model(initial_points)
            - initial_condition(initial_points[:, 0:1], initial_points[:, 1:2])
        ).abs().max().item()
    assert initial_error <= 10.0 * torch.finfo(dtype).eps
    return initial_error


# initial_error = validate_setup(model, device, dtype, early_time_fraction)
# print(
#     f"device={device}, dtype={dtype}, "
#     f"parameters={sum(p.numel() for p in model.parameters()):,}"
# )
# print(f"maximum initial-condition error: {initial_error:.3e}")


In [ ]:
history: list[dict[str, float]] = []
best_loss = math.inf
best_epoch = 0
best_losses: dict[str, float] = {}
best_state: dict[str, Tensor] | None = None

for epoch in range(1, epochs + 1):
    model.train()
    interior = sample_interior(
        interior_batch, device, dtype, early_time_fraction
    )
    boundary, boundary_normals = sample_boundary(
        boundary_batch, device, dtype, early_time_fraction
    )
    residual = pde_residual(model, interior)
    flux = boundary_flux(model, boundary, boundary_normals)
    pde_loss = residual.square().mean()
    boundary_loss = flux.square().mean()
    total_loss = pde_loss + boundary_weight * boundary_loss

    if not torch.isfinite(total_loss):
        raise RuntimeError(f"Non-finite loss at epoch {epoch}")
    optimizer.zero_grad(set_to_none=True)
    total_loss.backward()
    gradient_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    if not torch.isfinite(gradient_norm):
        raise RuntimeError(f"Non-finite gradient at epoch {epoch}")
    optimizer.step()
    scheduler.step()

    losses = {
        "total_loss": float(total_loss.detach().cpu()),
        "pde_loss": float(pde_loss.detach().cpu()),
        "boundary_loss": float(boundary_loss.detach().cpu()),
    }
    row = {
        "epoch": epoch,
        **losses,
        "learning_rate": scheduler.get_last_lr()[0],
        "gradient_norm": float(gradient_norm.detach().cpu()),
    }
    history.append(row)
    if losses["total_loss"] < best_loss:
        best_loss = losses["total_loss"]
        best_epoch = epoch
        best_losses = losses
        best_state = {
            name: value.detach().cpu().clone()
            for name, value in model.state_dict().items()
        }

    if epoch == 1 or epoch % log_every == 0 or epoch == epochs:
        print(
            f"epoch {epoch:6d}/{epochs}  total={losses['total_loss']:.4e}  "
            f"pde={losses['pde_loss']:.4e}  boundary={losses['boundary_loss']:.4e}  "
            f"lr={row['learning_rate']:.2e}"
        )


In [ ]:
epochs_history = [row["epoch"] for row in history]

plt.figure(figsize=(9, 5))

for loss_name in ("total_loss", "pde_loss", "boundary_loss"):
    plt.plot(
        epochs_history,
        [row[loss_name] for row in history],
        label=loss_name,
        linewidth=1.5,
    )

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.yscale("log")  # Useful because the losses differ greatly in magnitude
plt.title("PINN training losses")
plt.grid(True, which="both", alpha=0.3)
plt.legend()
plt.tight_layout()
# plt.savefig("pinn_losses_02_015.png", dpi=300)

In [ ]:
final_losses = {key: history[-1][key]
                    for key in ("total_loss", "pde_loss", "boundary_loss")}

In [ ]:
results_dir = Path("comparison_results")
points_xy = np.atleast_2d(
    np.loadtxt(results_dir / "comparison_points.txt")
)

In [ ]:

# Optional: evaluate the best model found during training
# model.load_state_dict(best_state)

In [ ]:
snapshot_values = np.empty(
    (len(snapshot_times), len(points_xy)),
    dtype=np.float64,
)

In [ ]:
was_training = model.training
model.eval()

In [ ]:
with torch.inference_mode():
    for i, time in enumerate(snapshot_times):
        # PINN inputs have columns (x, y, t)
        points_xyt = np.column_stack((
            points_xy,
            np.full(len(points_xy), time),
        ))

        inputs = torch.as_tensor(
            points_xyt,
            device=device,
            dtype=dtype,
        )

        snapshot_values[i] = (
            model(inputs)
            .squeeze(-1)
            .cpu()
            .numpy()
        )

model.train(was_training)

In [ ]:
output_file = results_dir / "pinn_comparison_snapshots.npz"

np.savez_compressed(
    output_file,
    points=points_xy,             # (n_points, 2)
    times=snapshot_times,         # (n_times,)
    u=snapshot_values,            # (n_times, n_points)
)

print(f"Saved PINN results to {output_file}")